In [3]:
import pandas as pd
cols = ["query", "subject", "pident", "length", "mismatch", "gapopen", 
        "qstart", "qend", "sstart", "send", "evalue", "bitscore"]

fwd = pd.read_csv("baro_vs_koda.txt", sep="\t", header=None, names=cols)
rev = pd.read_csv("koda_vs_baro.txt", sep="\t", header=None, names=cols)
fwdbest = dict(zip(fwd["query"], fwd["subject"]))
revbest = dict(zip(rev["query"], rev["subject"]))
rbh = []
for baro, koda in fwdbest.items():
    if revbest.get(koda) == baro:
        rbh.append({"barophilus": baro, "kodakarensis": koda})

rbhdf = pd.DataFrame(rbh)
print(f"Reciprocal best hits: {len(rbhdf)}")
rbhdf.to_csv("orthologpairs.csv", index=False)

Reciprocal best hits: 1539


In [15]:
import requests, os, time
rbh = pd.read_csv("orthologpairs.csv")
def downloadstructures(accessions, output):
    os.makedirs(output, exist_ok=True)
    found, missing = 0, 0
    for acc in accessions:
        accclean = acc.split("|")[1] if "|" in acc else acc
        pdbpath = os.path.join(output, f"AF-{accclean}-F1.pdb")
        if os.path.exists(pdbpath):
            found += 1
            continue
        url = f"https://alphafold.ebi.ac.uk/files/AF-{accclean}-F1-model_v6.pdb"
        r = requests.get(url)
        if r.status_code == 200:
            with open(pdbpath, "wb") as f:
                f.write(r.content)
            found += 1
        else:
            missing += 1
            print(f"Missing: {accclean} ({r.status_code})")
        time.sleep(0.2)
    print(f"Done: {found} found, {missing} missing")

downloadstructures(rbh["barophilus"], "structures/barophilus")
downloadstructures(rbh["kodakarensis"], "structures/kodakarensis")

Missing: F0LKL3 (404)
Missing: F0LH94 (404)
Missing: F0LJB0 (404)
Missing: F0LMD0 (404)
Done: 1535 found, 4 missing
Missing: Q5JHB6 (404)
Missing: Q5JJ43 (404)
Missing: A0ACM8PXE4 (404)
Missing: Q5JE00 (404)
Missing: A0ACM8PXE9 (404)
Missing: A0ACM8PXE6 (404)
Missing: A0ACM8PXE3 (404)
Done: 1532 found, 7 missing


In [12]:
from Bio import SeqIO
for record in SeqIO.parse(os.path.expanduser("~/piezo-project/data/barophilus.fasta.txt"), "fasta"):
    print(record.id)
    break

sp|F0LJX3|LHRC2_THEBM


In [14]:
import requests
r = requests.get("https://alphafold.ebi.ac.uk/api/prediction/F0LJX3")
print(r.status_code)
if r.status_code == 200:
    print(r.json()[0]["pdbUrl"])


200
https://alphafold.ebi.ac.uk/files/AF-F0LJX3-F1-model_v6.pdb


In [21]:
baro = set(f.split("-")[1] for f in os.listdir("structures/barophilus") if f.startswith("AF-"))
koda = set(f.split("-")[1] for f in os.listdir("structures/kodakarensis") if f.startswith("AF-"))

rbh["baroacc"] = rbh["barophilus"].apply(lambda x: x.split("|")[1] if "|" in x else x)
rbh["kodaacc"] = rbh["kodakarensis"].apply(lambda x: x.split("|")[1] if "|" in x else x)
both = rbh[(rbh["baroacc"].isin(baro)) & (rbh["kodaacc"].isin(koda))]
print(f"Both: {len(both)}")
both.to_csv("ortholog_pairs_with_structures.csv", index=False)

Both: 1528


In [22]:
import pandas as pd
cols = ["query", "subject", "pident", "length", "qstart", "qend",
        "sstart", "send", "qlen", "slen", "evalue", "bitscore"]
fwd = pd.read_csv("baro_vs_koda_v2.txt", sep="\t", header=None, names=cols)
rev = pd.read_csv("koda_vs_baro_v2.txt", sep="\t", header=None, names=cols)
print(fwd["query"].nunique(), "barophilus proteins with a hit")
print(rev["query"].nunique(), "kodakarensis proteins with a hit")

fwdbest = fwd.sort_values("bitscore", ascending=False).drop_duplicates(subset="query", keep="first")
revbest = rev.sort_values("bitscore", ascending=False).drop_duplicates(subset="query", keep="first")

1742 barophilus proteins with a hit
1703 kodakarensis proteins with a hit


In [36]:
fwdmap = dict(zip(fwdbest["query"], fwdbest["subject"]))
revmap = dict(zip(revbest["query"], revbest["subject"]))
pairs = []
for baro, koda in fwdmap.items():
    if revmap.get(koda) == baro:
        pairs.append((baro, koda))
rbh = pd.DataFrame(pairs, columns=["barophilus", "kodakarensis"])
rbh = rbh.merge(fwdbest, left_on=["barophilus", "kodakarensis"], right_on=["query", "subject"])
rbh["qcov"] = (rbh["qend"] - rbh["qstart"] + 1) / rbh["qlen"]
rbh["scov"] = (rbh["send"] - rbh["sstart"] + 1) / rbh["slen"]

filtered = rbh[(rbh["qcov"] >= 0.8) & (rbh["scov"] >= 0.8)]
print(len(rbh), "reciprocal pairs before coverage filter")
print(len(filtered), "pairs after 80% coverage filter")

1539 reciprocal pairs before coverage filter
1440 pairs after 80% coverage filter


In [37]:
old = pd.read_csv("orthologpairs.csv")
oldset = set(zip(old["barophilus"], old["kodakarensis"]))
newset = set(zip(filtered["barophilus"], filtered["kodakarensis"]))
print(len(oldset & newset), "pairs in both old and new")
print(len(newset - oldset), "new pairs")
print(len(oldset - newset), "old pairs dropped")
filtered[["barophilus", "kodakarensis", "pident", "qcov", "scov"]].to_csv("orthologpairsv2.csv", index=False)

1440 pairs in both old and new
0 new pairs
99 old pairs dropped


In [38]:
withstructs = pd.read_csv("orthologpairswithstructures.csv")
final = filtered.merge(withstructs[["barophilus", "kodakarensis"]], on=["barophilus", "kodakarensis"])
print(len(final), "pairs with 80% coverage and structures on both sides")
final.to_csv("orthologpairsfinal.csv", index=False)

1435 pairs with 80% coverage and structures on both sides


In [39]:
cov90 = final[(final["qcov"] >= 0.9) & (final["scov"] >= 0.9)]
print(len(cov90), "pairs at 90% coverage with structures")

1341 pairs at 90% coverage with structures
